# SAMap supplementary figure, panel f: transferred pseudotime in the human sections, observed and shuffled

The human sections colored by transferred pseudotime: the transfer of record (left), and one transfer each after
shuffling mouse pseudotime across all mouse nuclei (middle) or only within each mouse compartment (right), before
the same SAMap transfer. Rows are the two human phases; each phase is one physical section whose three 10x lanes
share one coordinate frame, so they are drawn together. All maps share one scale (210 µm per mm) and the fixed 0-1
color scale, so a shuffle that flattens the transferred values shows as a flat map. Crops and rotations are those of
Fig. 6E. Every positioned nucleus is drawn.

**Inputs**: `human_cells.csv` (rule `samap_figure_tables`: QC'd Slide-tags positions `x_um_qc`/`y_um_qc`, phase,
the transfer of record) and `example_transfers.csv` (rule `samap_ptnull`: the observed transfer
and one transfer per shuffle design). The notebook stops unless the permutation run's observed transfer equals the
transfer of record.
**Outputs**, in `OUT`: `sfig_samap_panel_f.png` (300 dpi), `sfig_samap_panel_f_600dpi.png`,
`sfig_samap_panel_f.svg` (text kept as text) and `sfig_samap_panel_f.provenance.json`.
`sfig_samap_compose.ipynb` places the panel in the figure and adds its letter.

**Running it.** Run all cells with this directory as the working directory: the shared modules next to the notebook
are imported from there. The cells above *Run* only define constants and functions. Paths are relative to the
repository root. Existing outputs are not overwritten unless `FORCE = True`.

The pipeline runs it with papermill (`workflow/rules/10c_samap_figures.smk`). By hand, from the repository root:

```bash
papermill notebooks/samap/sfig_samap_f_sections_shuffled.ipynb \
    results/notebooks/samap/sfig_samap_panel_f.executed.ipynb \
    --cwd notebooks/samap -p FORCE True
```

Commit this notebook without outputs: the provenance record notes whether the notebook matches its commit.

In [ ]:
# Parameters. papermill -p NAME VALUE overrides them in a cell below this one.
# Paths are relative to the repository root; REPO = None finds the root with git from the working directory.
REPO = None
HUMAN_CELLS = "results/samap_figures/tables/human_cells.csv"
EXAMPLE_TRANSFERS = "results/samap_figures/ptnull/example_transfers.csv"
OUT = "results/figures/samap_supp/panels"
FORCE = False  # overwrite existing outputs
NOTEBOOK = "notebooks/samap/sfig_samap_f_sections_shuffled.ipynb"  # this notebook, for the provenance record

In [ ]:
import logging
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from IPython.display import Image, display

import sfig_samap_style as style

# This notebook's messages in the cell output; other libraries stay at WARNING.
logging.basicConfig(format="%(message)s", stream=sys.stdout, force=True)
LOG = logging.getLogger("sfig_samap_f")
LOG.setLevel(logging.INFO)

## Constants

In [ ]:
LETTER = "f"
TITLE = "Shuffled mouse pseudotime keeps at most\nthe fibroblast/decidual split"
WIDTH_SHOWN = 520  # px, the PNG shown in this notebook

# Rows: crop (x and y limits in µm, None for the whole section), counter-clockwise rotation (degrees) and whether
# the rotated section is mirrored left-right about its center, per phase, as in Fig. 6E.
PHASES = {
    "menstrual": {"label": "menstrual", "limits": None, "rotation": 90, "mirror": True},
    "secretory": {"label": "secretory\n(decidualized)", "limits": ((400, 5500), (750, 5250)), "rotation": -90,
                  "mirror": True},
}

# Columns: example_transfers.csv column and header.
MAPS = {
    "observed": "observed",
    "uniform_example": "shuffled across\nall mouse cells",
    "within_compartment_example": "shuffled within\ncompartment",
}

UM_PER_MM = 210          # one scale for every map, so the two phases keep their true size ratio
SECTION_PAD_UM = 60      # keeps edge dots from being clipped
MAP_DOT = 0.45           # pt^2
NO_VALUE = "#b0b0b0"     # nuclei without a valued mouse neighbor; not imputed
LABEL_WIDTH_MM = 9.0     # row labels left of the maps
MAP_GAP_MM = 2.0
SCALE_BAR_UM = 1000

## Sections

In [ ]:
def rotate(x, y, degrees, cx, cy):
    """Rotate points counter-clockwise by `degrees` about (cx, cy)."""
    t = np.deg2rad(degrees)
    return (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx, (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy


def section_coords(human: pd.DataFrame, phase: str, spec: dict):
    """Nucleus ids and coordinates (µm) drawn for one phase, rotated and mirrored as `spec` says: positioned and in
    the crop."""
    is_drawn = (human.timepoint == phase) & human.x_um_qc.notna()
    cells = human.loc[is_drawn, ["x_um_qc", "y_um_qc"]]
    x, y = cells.x_um_qc.to_numpy(), cells.y_um_qc.to_numpy()
    if spec["limits"] is not None:
        (x0, x1), (y0, y1) = spec["limits"]
        keep = (x >= x0) & (x <= x1) & (y >= y0) & (y <= y1)
        cells, x, y = cells[keep], x[keep], y[keep]
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    else:
        cx, cy = (x.min() + x.max()) / 2, (y.min() + y.max()) / 2
    x_rotated, y_rotated = rotate(x, y, spec["rotation"], cx, cy)
    if spec["mirror"]:
        x_rotated = 2 * cx - x_rotated  # left-right, about the center the section was rotated around
    return cells.index, x_rotated, y_rotated

## Figure

In [ ]:
def draw_sections(sections: dict, transfers: pd.DataFrame):
    fig, width_mm, height_mm = style.new_panel(LETTER)

    # Every map at UM_PER_MM; each column is as wide as the wider section, and the grid is centered right of the
    # row labels, below the two-line title and the two-line headers.
    extent = {
        phase: ((x.min() - SECTION_PAD_UM, x.max() + SECTION_PAD_UM), (y.min() - SECTION_PAD_UM, y.max() + SECTION_PAD_UM))
        for phase, (_, x, y) in sections.items()
    }
    size_mm = {phase: ((xs[1] - xs[0]) / UM_PER_MM, (ys[1] - ys[0]) / UM_PER_MM) for phase, (xs, ys) in extent.items()}
    col_w = max(w for w, _ in size_mm.values())
    grid_w = 3 * col_w + 2 * MAP_GAP_MM
    x_start = LABEL_WIDTH_MM + (width_mm - LABEL_WIDTH_MM - grid_w) / 2
    maps_top = height_mm - 17.0

    y_top = maps_top
    for row, (phase, spec) in enumerate(PHASES.items()):
        ids, x, y = sections[phase]
        (xs, ys), (w_mm, h_mm) = extent[phase], size_mm[phase]
        y_bottom = y_top - h_mm
        for col, (column, header) in enumerate(MAPS.items()):
            cell_left = x_start + col * (col_w + MAP_GAP_MM)
            ax = fig.add_axes([(cell_left + (col_w - w_mm) / 2) / width_mm, y_bottom / height_mm,
                               w_mm / width_mm, h_mm / height_mm])
            ax.set_xlim(*xs)
            ax.set_ylim(*ys)
            ax.axis("off")

            # Nuclei without a valued mouse neighbor go underneath, in grey; they are not imputed.
            values = transfers.loc[ids, column].to_numpy()
            missing = np.isnan(values)
            ax.scatter(x[missing], y[missing], s=MAP_DOT, color=NO_VALUE, linewidths=0, rasterized=True)
            colors = style.PSEUDOTIME_CMAP(style.PSEUDOTIME_NORM(values[~missing]))
            ax.scatter(x[~missing], y[~missing], s=MAP_DOT, c=colors, linewidths=0, rasterized=True)
            if row == 0:
                fig.text((cell_left + col_w / 2) / width_mm, (maps_top + 1.0) / height_mm, header,
                         ha="center", va="bottom", fontsize=8)
        fig.text((x_start - 3.0) / width_mm, (y_bottom + h_mm / 2) / height_mm, spec["label"], rotation=90,
                 ha="center", va="center", fontsize=8)
        y_top = y_bottom - MAP_GAP_MM

    # Shared colorbar under the middle column, scale bar under the right one.
    cax = fig.add_axes([(x_start + col_w + MAP_GAP_MM - 6.0) / width_mm, 5.2 / height_mm,
                        (col_w + 12.0) / width_mm, 1.6 / height_mm])
    bar = fig.colorbar(mpl.cm.ScalarMappable(norm=style.PSEUDOTIME_NORM, cmap=style.PSEUDOTIME_CMAP), cax=cax,
                       orientation="horizontal", ticks=[0, 0.5, 1])
    bar.ax.set_xticklabels(["0", "0.5", "1"])
    bar.outline.set_visible(False)
    bar.ax.tick_params(length=1.5, pad=1.5)
    cax.set_title("transferred pseudotime", fontsize=8, loc="center", pad=3)

    bar_mm = SCALE_BAR_UM / UM_PER_MM
    bar_right = x_start + grid_w - 2.0
    fig.add_artist(Line2D([(bar_right - bar_mm) / width_mm, bar_right / width_mm], [6.0 / height_mm, 6.0 / height_mm],
                          color="black", lw=1.0))
    fig.text((bar_right - bar_mm / 2) / width_mm, 7.0 / height_mm, "1 mm", ha="center", va="bottom", fontsize=7)
    style.panel_title(fig, width_mm, height_mm, TITLE)
    return fig

## Run

In [ ]:
repo = style.find_repo(REPO)
notebook = repo / NOTEBOOK
if not notebook.is_file():
    raise FileNotFoundError(f"{notebook} not found; set NOTEBOOK to this notebook's path")

paths = style.panel_paths(repo / OUT, LETTER)
style.refuse_overwrite(paths.values(), FORCE)
paths["png"].parent.mkdir(parents=True, exist_ok=True)

In [ ]:
human = pd.read_csv(repo / HUMAN_CELLS, index_col="cell")
transfers = pd.read_csv(repo / EXAMPLE_TRANSFERS).set_index("cell")
style.check_same_transfer(transfers, human)

sections = {phase: section_coords(human, phase, spec) for phase, spec in PHASES.items()}
drawn = {
    phase: {"nuclei": len(ids), "no_value": int(transfers.loc[ids, "observed"].isna().sum())}
    for phase, (ids, _, _) in sections.items()
}
LOG.info("Nuclei drawn per section: %s", drawn)

In [ ]:
font = style.choose_font()
style.apply_style(font)
LOG.info("Font: %s", font)

fig = draw_sections(sections, transfers)

style.save_panel(fig, LETTER, paths)
plt.close(fig)
LOG.info("Wrote %s", [str(path) for name, path in paths.items() if name != "provenance"])

# The saved PNG rather than the live figure, so what shows here is the file itself.
display(Image(filename=str(paths["png"]), width=WIDTH_SHOWN))

In [ ]:
record = style.provenance(
    notebook,
    parameters={
        "REPO": str(repo),
        "HUMAN_CELLS": HUMAN_CELLS,
        "EXAMPLE_TRANSFERS": EXAMPLE_TRANSFERS,
        "OUT": OUT,
        "FORCE": FORCE,
    },
    inputs={"human_cells": repo / HUMAN_CELLS, "example_transfers": repo / EXAMPLE_TRANSFERS},
    outputs=[paths["png"], paths["png_600dpi"], paths["svg"]],
    font=font,
)
record["panel"] = style.panel_spec(LETTER)
record["spatial_nuclei_drawn"] = drawn
record["sections"] = {
    phase: {"limits_um": spec["limits"], "rotation_deg": spec["rotation"], "mirrored_left_right": spec["mirror"]}
    for phase, spec in PHASES.items()
}
style.write_json(paths["provenance"], record)
LOG.info("Wrote %s", paths["provenance"])